# ⚖️ Model Comparison and Benchmarking

This notebook provides a comprehensive comparison of different machine learning models and approaches available in the Data Science Portfolio.

## 🎯 Objectives

1. Compare traditional ML vs. deep learning models
2. Benchmark performance across different algorithms
3. Analyze trade-offs between accuracy and interpretability
4. Evaluate computational efficiency
5. Provide model selection guidelines

## 📊 Comparison Dimensions

- **Accuracy**: Predictive performance metrics
- **Speed**: Training and inference time
- **Interpretability**: Model explainability
- **Scalability**: Performance with large datasets
- **Robustness**: Handling of noise and outliers

## Setup and Data Preparation

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# ML models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Evaluation
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, make_classification
)
from sklearn.preprocessing import StandardScaler

# Portfolio modules
from modern_bank_churn.evaluation_enhancements import ModelEvaluator

print("✅ Libraries loaded successfully!")

In [ ]:
# Generate synthetic datasets with different characteristics
np.random.seed(42)

datasets = {}

# 1. Linearly separable data
X_linear, y_linear = make_classification(
    n_samples=5000, n_features=20, n_informative=15,
    n_redundant=5, n_clusters_per_class=1,
    flip_y=0.05, random_state=42
)
datasets['Linear'] = (X_linear, y_linear)

# 2. Non-linear complex data
X_complex, y_complex = make_classification(
    n_samples=5000, n_features=20, n_informative=10,
    n_redundant=5, n_clusters_per_class=3,
    flip_y=0.1, random_state=42
)
datasets['Complex'] = (X_complex, y_complex)

# 3. High-dimensional data
X_highdim, y_highdim = make_classification(
    n_samples=2000, n_features=100, n_informative=20,
    n_redundant=30, n_clusters_per_class=2,
    flip_y=0.1, random_state=42
)
datasets['High-Dim'] = (X_highdim, y_highdim)

# 4. Imbalanced data
X_imbalanced, y_imbalanced = make_classification(
    n_samples=5000, n_features=20, n_informative=15,
    n_redundant=5, weights=[0.9, 0.1],
    flip_y=0.05, random_state=42
)
datasets['Imbalanced'] = (X_imbalanced, y_imbalanced)

print("📊 Datasets created:")
for name, (X, y) in datasets.items():
    print(f"  • {name}: {X.shape[0]} samples, {X.shape[1]} features, {y.mean():.2%} positive class")

## Model Comparison Framework

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
    'SVM': SVC(probability=True, random_state=42),
    'Naive Bayes': GaussianNB(),
    'Neural Network': MLPClassifier(hidden_layers=(100, 50), max_iter=1000, random_state=42)
}

print(f"🤖 Models to compare: {len(models)}")
for name in models.keys():
    print(f"  • {name}")

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    """
    Comprehensive model evaluation.
    """
    # Training time
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Inference time
    start_time = time.time()
    y_pred = model.predict(X_test)
    inference_time = (time.time() - start_time) / len(X_test) * 1000  # ms per sample
    
    # Predictions
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    # Metrics
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='binary'),
        'recall': recall_score(y_test, y_pred, average='binary'),
        'f1': f1_score(y_test, y_pred, average='binary'),
        'auc_roc': roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else 0,
        'training_time': training_time,
        'inference_time': inference_time
    }
    
    return metrics

# Run comparison
results = {}

for dataset_name, (X, y) in datasets.items():
    print(f"\n📊 Evaluating on {dataset_name} dataset...")
    
    # Split and scale data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    dataset_results = {}
    
    for model_name, model in models.items():
        try:
            metrics = evaluate_model(
                model.__class__(**model.get_params()),
                X_train_scaled, X_test_scaled,
                y_train, y_test
            )
            dataset_results[model_name] = metrics
            print(f"  ✓ {model_name}: AUC={metrics['auc_roc']:.3f}, Time={metrics['training_time']:.2f}s")
        except Exception as e:
            print(f"  ✗ {model_name}: Error - {str(e)[:50]}")
            dataset_results[model_name] = {metric: 0 for metric in 
                                         ['accuracy', 'precision', 'recall', 'f1', 'auc_roc',
                                          'training_time', 'inference_time']}
    
    results[dataset_name] = dataset_results

print("\n✅ Evaluation complete!")

## Performance Visualization

In [ ]:
# Create comparison dataframe
comparison_data = []

for dataset_name, dataset_results in results.items():
    for model_name, metrics in dataset_results.items():
        row = {
            'Dataset': dataset_name,
            'Model': model_name,
            **metrics
        }
        comparison_data.append(row)

df_comparison = pd.DataFrame(comparison_data)

# Performance heatmap
plt.figure(figsize=(14, 8))

# Pivot for heatmap
pivot_auc = df_comparison.pivot(index='Model', columns='Dataset', values='auc_roc')

sns.heatmap(pivot_auc, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.5, vmax=1.0, cbar_kws={'label': 'AUC-ROC'})
plt.title('Model Performance Comparison (AUC-ROC)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Training time comparison
plt.figure(figsize=(14, 6))

pivot_time = df_comparison.pivot(index='Model', columns='Dataset', values='training_time')

ax = pivot_time.plot(kind='bar', width=0.8)
plt.title('Training Time Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Training Time (seconds)')
plt.yscale('log')
plt.legend(title='Dataset', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Radar chart for multi-metric comparison
import plotly.graph_objects as go

# Select best performing models
avg_performance = df_comparison.groupby('Model')['auc_roc'].mean().sort_values(ascending=False)
top_models = avg_performance.head(5).index.tolist()

# Prepare data for radar chart
metrics_radar = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']

fig = go.Figure()

for model in top_models:
    model_data = df_comparison[df_comparison['Model'] == model]
    avg_metrics = model_data[metrics_radar].mean()
    
    fig.add_trace(go.Scatterpolar(
        r=avg_metrics.values,
        theta=metrics_radar,
        fill='toself',
        name=model
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1]
        )),
    showlegend=True,
    title="Top 5 Models - Multi-Metric Comparison",
    height=500
)

fig.show()

## Trade-off Analysis

In [ ]:
# Accuracy vs Speed trade-off
avg_metrics = df_comparison.groupby('Model').agg({
    'auc_roc': 'mean',
    'training_time': 'mean',
    'inference_time': 'mean'
}).reset_index()

# Add interpretability score (subjective)
interpretability = {
    'Logistic Regression': 9,
    'Decision Tree': 8,
    'Naive Bayes': 8,
    'Random Forest': 5,
    'Gradient Boosting': 4,
    'XGBoost': 4,
    'LightGBM': 4,
    'SVM': 3,
    'Neural Network': 2
}

avg_metrics['interpretability'] = avg_metrics['Model'].map(interpretability)

# Create scatter plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Accuracy vs Training Time
ax1 = axes[0]
scatter1 = ax1.scatter(avg_metrics['training_time'], avg_metrics['auc_roc'],
                       s=avg_metrics['interpretability']*30,
                       c=avg_metrics['interpretability'],
                       cmap='viridis', alpha=0.7)

for idx, row in avg_metrics.iterrows():
    ax1.annotate(row['Model'], (row['training_time'], row['auc_roc']),
                fontsize=8, ha='right')

ax1.set_xlabel('Training Time (seconds)')
ax1.set_ylabel('AUC-ROC Score')
ax1.set_title('Accuracy vs Training Time Trade-off')
ax1.grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=ax1, label='Interpretability')

# Accuracy vs Interpretability
ax2 = axes[1]
ax2.scatter(avg_metrics['interpretability'], avg_metrics['auc_roc'],
           s=100, alpha=0.7)

for idx, row in avg_metrics.iterrows():
    ax2.annotate(row['Model'], (row['interpretability'], row['auc_roc']),
                fontsize=8, ha='left', rotation=15)

ax2.set_xlabel('Interpretability Score')
ax2.set_ylabel('AUC-ROC Score')
ax2.set_title('Accuracy vs Interpretability Trade-off')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Trade-off Analysis Summary:")
print(avg_metrics[['Model', 'auc_roc', 'training_time', 'interpretability']].to_string(index=False))

## Statistical Significance Testing

In [ ]:
from scipy import stats
from statistical_methods.hypothesis_tester import HypothesisTester

# Cross-validation for robust comparison
print("🔬 Statistical Significance Testing (5-fold CV)\n")

# Use Complex dataset for testing
X, y = datasets['Complex']
X_scaled = StandardScaler().fit_transform(X)

cv_scores = {}
for model_name, model in list(models.items())[:5]:  # Top 5 models
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='roc_auc')
    cv_scores[model_name] = scores
    print(f"{model_name}: {scores.mean():.3f} ± {scores.std():.3f}")

# Pairwise comparisons
tester = HypothesisTester(alpha=0.05, correction_method='bonferroni')

print("\n📊 Pairwise Model Comparisons:")
comparison_results = []

model_names = list(cv_scores.keys())
for i in range(len(model_names)):
    for j in range(i+1, len(model_names)):
        model1, model2 = model_names[i], model_names[j]
        scores1, scores2 = cv_scores[model1], cv_scores[model2]
        
        # Paired t-test (same CV folds)
        result = tester.t_test(scores1, scores2, paired=True)
        
        comparison_results.append({
            'Model 1': model1,
            'Model 2': model2,
            'Diff': scores1.mean() - scores2.mean(),
            'p-value': result['p_value'],
            'Significant': result['p_value'] < 0.05
        })

df_comparisons = pd.DataFrame(comparison_results)
print(df_comparisons.to_string(index=False))

## Model Selection Guidelines

In [ ]:
# Create recommendation matrix
recommendations = {
    'Scenario': [
        'Linear relationships',
        'Non-linear complex patterns',
        'High-dimensional data',
        'Imbalanced datasets',
        'Real-time predictions',
        'Interpretability required',
        'Large datasets (>1M rows)',
        'Limited training time',
        'Maximum accuracy needed'
    ],
    'Recommended Model': [
        'Logistic Regression',
        'XGBoost / Neural Network',
        'Random Forest / LightGBM',
        'XGBoost with class weights',
        'LightGBM / Logistic Regression',
        'Decision Tree / Logistic Regression',
        'LightGBM',
        'Naive Bayes / Logistic Regression',
        'Ensemble (XGBoost + LightGBM + NN)'
    ],
    'Reason': [
        'Simple, fast, and effective for linear data',
        'Captures complex non-linear relationships',
        'Handles many features well, robust to noise',
        'Built-in support for class imbalance',
        'Fast inference, low memory footprint',
        'Easy to understand and explain',
        'Efficient memory usage and parallel processing',
        'Fastest training time',
        'Combining models improves performance'
    ]
}

df_recommendations = pd.DataFrame(recommendations)

print("📋 Model Selection Guidelines\n")
print("=" * 80)
for idx, row in df_recommendations.iterrows():
    print(f"\n🎯 {row['Scenario']}")
    print(f"   Model: {row['Recommended Model']}")
    print(f"   Reason: {row['Reason']}")

## Ensemble Model Creation

In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

# Create ensemble models
X, y = datasets['Complex']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Voting ensemble
voting_ensemble = VotingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')),
        ('lgbm', LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)),
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
    ],
    voting='soft'
)

# Stacking ensemble
stacking_ensemble = StackingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=50, random_state=42, eval_metric='logloss')),
        ('lgbm', LGBMClassifier(n_estimators=50, random_state=42, verbose=-1)),
        ('rf', RandomForestClassifier(n_estimators=50, random_state=42))
    ],
    final_estimator=LogisticRegression(random_state=42),
    cv=5
)

# Train and evaluate ensembles
ensemble_results = {}

for name, ensemble in [('Voting', voting_ensemble), ('Stacking', stacking_ensemble)]:
    # Train
    start_time = time.time()
    ensemble.fit(X_train_scaled, y_train)
    training_time = time.time() - start_time
    
    # Predict
    y_pred = ensemble.predict(X_test_scaled)
    y_pred_proba = ensemble.predict_proba(X_test_scaled)[:, 1]
    
    # Evaluate
    ensemble_results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'auc_roc': roc_auc_score(y_test, y_pred_proba),
        'f1': f1_score(y_test, y_pred),
        'training_time': training_time
    }

print("🎭 Ensemble Model Performance:\n")
df_ensemble = pd.DataFrame(ensemble_results).T
print(df_ensemble.round(3))

# Compare with individual models
individual_scores = {
    'XGBoost': models['XGBoost'].fit(X_train_scaled, y_train).score(X_test_scaled, y_test),
    'LightGBM': models['LightGBM'].fit(X_train_scaled, y_train).score(X_test_scaled, y_test),
    'Random Forest': models['Random Forest'].fit(X_train_scaled, y_train).score(X_test_scaled, y_test)
}

print("\n📊 Improvement over individual models:")
best_individual = max(individual_scores.values())
for name, results in ensemble_results.items():
    improvement = (results['accuracy'] - best_individual) * 100
    print(f"{name} Ensemble: {improvement:+.2f}% improvement")

## Performance Summary Dashboard

In [ ]:
import plotly.subplots as sp
import plotly.graph_objects as go

# Create comprehensive dashboard
fig = sp.make_subplots(
    rows=2, cols=2,
    subplot_titles=['Average Performance', 'Training Time', 
                   'Dataset Sensitivity', 'Ensemble Comparison'],
    specs=[[{'type': 'bar'}, {'type': 'scatter'}],
           [{'type': 'heatmap'}, {'type': 'bar'}]]
)

# 1. Average Performance
avg_perf = df_comparison.groupby('Model')['auc_roc'].mean().sort_values(ascending=False)
fig.add_trace(
    go.Bar(x=avg_perf.index, y=avg_perf.values, name='AUC-ROC'),
    row=1, col=1
)

# 2. Training Time vs Performance
fig.add_trace(
    go.Scatter(
        x=avg_metrics['training_time'],
        y=avg_metrics['auc_roc'],
        mode='markers+text',
        text=avg_metrics['Model'],
        textposition='top center',
        marker=dict(size=10),
        name='Models'
    ),
    row=1, col=2
)

# 3. Dataset Sensitivity Heatmap
sensitivity_data = pivot_auc.values
fig.add_trace(
    go.Heatmap(
        z=sensitivity_data,
        x=pivot_auc.columns,
        y=pivot_auc.index,
        colorscale='RdYlGn',
        text=np.round(sensitivity_data, 3),
        texttemplate='%{text}',
        textfont={"size": 8}
    ),
    row=2, col=1
)

# 4. Ensemble Comparison
ensemble_comparison = pd.DataFrame({
    'Individual Best': [best_individual],
    'Voting': [ensemble_results['Voting']['accuracy']],
    'Stacking': [ensemble_results['Stacking']['accuracy']]
})

for col in ensemble_comparison.columns:
    fig.add_trace(
        go.Bar(x=[col], y=ensemble_comparison[col].values, name=col),
        row=2, col=2
    )

# Update layout
fig.update_layout(
    height=800,
    showlegend=True,
    title_text="Model Comparison Dashboard",
    title_font_size=16
)

fig.update_xaxes(title_text="Model", row=1, col=1)
fig.update_yaxes(title_text="AUC-ROC", row=1, col=1)
fig.update_xaxes(title_text="Training Time (s)", row=1, col=2)
fig.update_yaxes(title_text="AUC-ROC", row=1, col=2)
fig.update_xaxes(title_text="Dataset", row=2, col=1)
fig.update_yaxes(title_text="Model", row=2, col=1)
fig.update_xaxes(title_text="Model Type", row=2, col=2)
fig.update_yaxes(title_text="Accuracy", row=2, col=2)

fig.show()

print("📊 Dashboard generated successfully!")

## Key Findings and Recommendations

In [ ]:
print("=" * 80)
print("📊 MODEL COMPARISON SUMMARY REPORT")
print("=" * 80)

print("\n🏆 Top Performers by Dataset:")
for dataset_name in results.keys():
    dataset_results = results[dataset_name]
    best_model = max(dataset_results.items(), key=lambda x: x[1]['auc_roc'])
    print(f"  • {dataset_name}: {best_model[0]} (AUC: {best_model[1]['auc_roc']:.3f})")

print("\n⚡ Fastest Models:")
speed_ranking = avg_metrics.nsmallest(3, 'training_time')[['Model', 'training_time']]
for idx, row in speed_ranking.iterrows():
    print(f"  {idx+1}. {row['Model']}: {row['training_time']:.3f}s")

print("\n🎯 Most Interpretable Models:")
interp_ranking = avg_metrics.nlargest(3, 'interpretability')[['Model', 'interpretability']]
for idx, row in interp_ranking.iterrows():
    print(f"  {idx+1}. {row['Model']}: Score {row['interpretability']}/10")

print("\n📈 Best Overall (Balanced):")
# Normalized scoring
avg_metrics['overall_score'] = (
    avg_metrics['auc_roc'] * 0.5 +  # 50% weight on accuracy
    (1 - avg_metrics['training_time'] / avg_metrics['training_time'].max()) * 0.3 +  # 30% on speed
    avg_metrics['interpretability'] / 10 * 0.2  # 20% on interpretability
)
best_overall = avg_metrics.nlargest(3, 'overall_score')[['Model', 'overall_score']]
for idx, row in best_overall.iterrows():
    print(f"  {idx+1}. {row['Model']}: {row['overall_score']:.3f}")

print("\n💡 Key Recommendations:")
print("  1. For production systems with real-time requirements: Use LightGBM or Logistic Regression")
print("  2. For maximum accuracy with sufficient resources: Use XGBoost or Ensemble methods")
print("  3. For regulatory/explainable AI requirements: Use Decision Trees or Logistic Regression")
print("  4. For proof-of-concepts and quick iterations: Use Random Forest (good balance)")
print("  5. For imbalanced datasets: Use XGBoost/LightGBM with class weights")

print("\n🔬 Statistical Insights:")
significant_pairs = df_comparisons[df_comparisons['Significant']]
if len(significant_pairs) > 0:
    print(f"  • {len(significant_pairs)} model pairs showed statistically significant differences")
    print(f"  • Largest significant difference: {significant_pairs['Diff'].abs().max():.3f}")
else:
    print("  • No statistically significant differences found (may need larger sample)")

print("\n" + "=" * 80)

## Conclusion

This comprehensive model comparison provides actionable insights for model selection:

### 🎯 Key Takeaways

1. **No Single Best Model**: Performance varies significantly across datasets
2. **Trade-offs Are Inevitable**: Accuracy vs Speed vs Interpretability
3. **Ensemble Methods Excel**: But at the cost of complexity and training time
4. **Context Matters**: Choose models based on specific requirements
5. **Validation Is Critical**: Always test on your specific data

### 📚 Further Reading

- [ML Pipeline Documentation](../docs/modules/ml_pipeline.md)
- [Statistical Methods](../docs/modules/statistics.md)
- [API Reference](../docs/api_reference.md)

---

**Note**: Results may vary based on data characteristics, hyperparameter tuning, and random seeds.